In [ ]:
# -*- coding: utf-8 -*-
import os
import gc
import json
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==============================================================================
# CONFIGURATION
# ==============================================================================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'

PREPROCESSED_H5 = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_50k_safe.h5'

def main():
    os.makedirs(os.path.dirname(PREPROCESSED_H5), exist_ok=True)
    
    # --------------------------------------------------------------------------
    # FASE 1: FILTERING (Pandas)
    # --------------------------------------------------------------------------
    print("[INFO] FASE 1: Memuat metadata...", flush=True)
    with open(ZHI_GENG_JSON, 'r') as f:
        zhi_geng_traces = set(json.load(f).keys())
        
    # Memuat CSV dengan sangat hemat memori
    df_raw = pd.read_csv(CSV_PATH, usecols=['trace_name', 'trace_category', 'p_arrival_sample'])
    df_unseen = df_raw[(df_raw['trace_category'].isin(['earthquake_local', 'noise'])) & 
                       (~df_raw['trace_name'].isin(zhi_geng_traces))]
    
    # Sampling 25K EQ + 25K Noise
    df_final = pd.concat([
        df_unseen[df_unseen['trace_category'] == 'earthquake_local'].sample(n=25000, random_state=42),
        df_unseen[df_unseen['trace_category'] == 'noise'].sample(n=25000, random_state=42)
    ]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    total_target = len(df_final)
    print(f"[INFO] Total data yang akan diproses: {total_target} sampel.", flush=True)
    
    # Bersihkan memori Pandas
    del df_raw, df_unseen
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 2: SEQUENTIAL EXTRACTION (HDF5)
    # --------------------------------------------------------------------------
    print(f"[INFO] FASE 2: Mengekstrak gelombang ke {PREPROCESSED_H5}...", flush=True)
    
    num_points = 700
    norm_points = 900
    berhasil = 0

    with h5py.File(PREPROCESSED_H5, 'w') as f_out:
        # Menyiapkan kerangka dataset kosong
        dset_1c = f_out.create_dataset("X_1C", shape=(total_target, 700, 1), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_3c = f_out.create_dataset("X_3C", shape=(total_target, 700, 3), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_y  = f_out.create_dataset("Y", shape=(total_target,), dtype=np.int32)
        dset_names = f_out.create_dataset("trace_name", shape=(total_target,), dtype=h5py.string_dtype(encoding='utf-8'))
        
        with h5py.File(HDF5_PATH, 'r') as f_in:
            data_group = f_in['data']
            
            # Loop sekuensial (Satu per satu, paling aman)
            for idx, row in tqdm(df_final.iterrows(), total=total_target, desc="Processing HDF5"):
                trace_id = row['trace_name']
                category = row['trace_category']
                p_arrival = 0 if pd.isna(row['p_arrival_sample']) else row['p_arrival_sample']
                
                if trace_id not in data_group:
                    continue
                    
                # Load wave ke RAM sesaat
                raw_wave = data_group[trace_id][()].astype(np.float32)
                raw_wave -= np.mean(raw_wave, axis=0) # Detrend
                
                if category == 'earthquake_local':
                    start = int(p_arrival)
                    if start < 0 or (start + norm_points) > len(raw_wave): 
                        del raw_wave
                        continue
                    wave = raw_wave[start:start+num_points, :]
                    norm_val = np.max(np.abs(raw_wave[start:start+norm_points, :]), axis=0)
                    label = 1
                else:
                    wave = raw_wave[:num_points, :]
                    norm_val = np.max(np.abs(raw_wave[:norm_points, :]), axis=0)
                    label = 0
                    
                norm_val[norm_val == 0] = 1e-8
                wave /= norm_val
                
                wave_1c = wave[:, 2].reshape(num_points, 1)
                
                # Tulis langsung ke disk
                dset_names[berhasil] = trace_id
                dset_3c[berhasil] = wave
                dset_1c[berhasil] = wave_1c
                dset_y[berhasil] = label
                berhasil += 1
                
                # Bersihkan variabel per iterasi
                del raw_wave, wave, wave_1c
                
                # Pembersihan paksa berkala
                if berhasil > 0 and berhasil % 2500 == 0:
                    gc.collect()
                    
        # Resizing file HDF5 output sesuai jumlah data yang benar-benar sukses terekstrak
        if berhasil < total_target:
            dset_1c.resize((berhasil, 700, 1))
            dset_3c.resize((berhasil, 700, 3))
            dset_y.resize((berhasil,))
            dset_names.resize((berhasil,))

    print("\n=======================================================")
    print(f" [SUKSES] Dataset {berhasil} sampel telah di-freeze secara Sekuensial!")
    print("=======================================================")

if __name__ == "__main__":
    main()

[INFO] FASE 1: Memuat metadata...
[INFO] Total data yang akan diproses: 50000 sampel.
[INFO] FASE 2: Mengekstrak gelombang ke /Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_50k_safe.h5...


Processing HDF5:   2%|▏         | 1226/50000 [03:25<4:02:43,  3.35it/s]Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x107c22290>
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/weakref.py", line 106, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 
Processing HDF5:   3%|▎         | 1295/50000 [03:46<4:15:30,  3.18it/s]